## 12. 外部 MCP：接入现成生态

> 来源：[Connect to external tools with MCP](https://code.claude.com/docs/en/agent-sdk/mcp)、[Scale to many tools with tool search](https://code.claude.com/docs/en/agent-sdk/tool-search)

除了进程内自定义工具，还能连**外部 MCP server**（数据库、浏览器、GitHub、Slack，社区有几百个现成的）。


### 12.1 三种 transport（选型规则：文档给命令 → stdio；给 URL → http/sse）

| 类型 | 配置形状 | 说明 |
|---|---|---|
| `stdio` | `{"command": "npx", "args": [...], "env": {...}}` | 本地子进程，`env` 传凭证 |
| `http` | `{"type": "http", "url": "...", "headers": {...}}` | 远程 server，`headers` 传认证头；代码配置只接受 `"http"`（`.mcp.json` 里 `"streamable-http"` 是别名） |
| `sse` | `{"type": "sse", "url": "...", "headers": {...}}` | 远程 SSE server |

远程 server 的最小接法（官方 quickstart 即这种形态；stdio 的完整可运行示例见本章末 demo cell）：

```python
options = ClaudeAgentOptions(
    mcp_servers={"docs": {"type": "http", "url": "https://mcp.example.com/mcp",
                          "headers": {"Authorization": f"Bearer {token}"}}},
    allowed_tools=["mcp__docs__*"],
)
```

配置有两条途径：代码里传 `mcp_servers={...}`，或项目根放 `.mcp.json`（`setting_sources` 含 `"project"` 时自动读取，`${VAR}` 语法运行时展开环境变量）。

### 12.2 授权与发现

- **MCP tool 要加进 `allowed_tools` 才免审批**；不在名单里的不是不能调，而是走权限流程（回调/规则/模式说了算，见 §9.1「权限评估顺序」）——但 headless 场景没配任何审批处理时表现就是"工具明明连上了却不用"。通配符只能作用在 tool 段：`mcp__github__*` 有效，`mcp__*` 会被忽略并告警。
- 授权粒度可以到单个 tool——对危险 server 用最小授权：

```python
mcp_servers={"postgres": {"command": "npx", "args": ["@mcp/postgres", DB_URL]}},
allowed_tools=["mcp__postgres__query"],   # 只放行只读查询，同 server 其他工具仍走审批
```

- **别靠放宽 permission mode 给 MCP 授权**：`acceptEdits` 不批 MCP tool（只批文件编辑与文件系统命令），`bypassPermissions` 又批得太宽——授权就用 `allowed_tools`（§9.2「permission_mode 全表」）。
- 发现 server 提供了什么 tool：读 init 消息的 `message.data["mcp_servers"]`，同时检查每个 server 的 `status` 字段是否为 `"connected"`（连接失败要在 agent 开工前发现）。默认连接超时 **30 秒**，可用 `env={"MCP_TIMEOUT": "60000"}`（毫秒）调高；调超时之外的三条缓解：换更轻量的 server、启动 agent 前先把 server 预热起来、查 server 自己的日志找初始化慢的原因。
- 连接失败的常见原因：stdio server 缺环境变量（查 `env` 字段）、`npx` 包不存在或 Node 不在 PATH、数据库连接串无效、远程 server 网络/防火墙不通。

### 12.3 认证

stdio server 用 `env` 字段传凭证；远程 server 用 `headers` 传 `Authorization`。MCP 规范支持 OAuth 2.1，但 **SDK 不代办 OAuth 流程**——应用自己完成 OAuth 拿到 access token，再放进 `headers`。


### 12.4 Tool search：工具多了怎么办

工具定义会吃 context（50 个 tool 约 10–20K token），且一次加载超过 30–50 个后模型选工具的准确率下降。**Tool search 默认开启**（除 Haiku 外的 Claude 模型都支持）：工具定义不预载，agent 只拿到摘要，需要时搜索目录、每次加载 3–5 个最相关的进 context；目录上限 10,000 个工具。

开关从子进程环境变量进，激活与否可以在消息流里直接观测到：

```python
options = ClaudeAgentOptions(
    mcp_servers={...},                     # 挂了很多 server 时才值得开
    env={"ENABLE_TOOL_SEARCH": "auto"},    # 开关经 env 传给 CLI 子进程
)
# 激活后的三个可见信号（观察代码同下方 demo cell）：
# 1. init 消息的 data["tools"] 清单里出现 "ToolSearch"
# 2. MCP tool 定义不再全量进 context，init 后 token 占用明显下降
# 3. 流里能看到 Claude 先调 ToolSearch 搜工具、再调搜出来的具体 mcp__ tool
```

`ENABLE_TOOL_SEARCH` 的取值：

| 值 | 行为 |
|---|---|
| 未设置 | 开启。两种环境自动回退为全量预载：Google Cloud 的 Agent Platform（仅 Sonnet 4.5+ / Opus 4.5+ 支持 tool search）、`ANTHROPIC_BASE_URL` 指向非第一方 host（多数代理不转发 tool search 依赖的 `tool_reference` block） |
| `"true"` | 强制开——在上述不支持的环境里请求会**直接失败**，不是回退 |
| `"false"` | 强制关，全量预载 |
| `"auto"` | 工具定义合计超过 context 的 10% 才激活；合计口径含全部 server（远程 MCP 和进程内 SDK MCP 都算） |
| `"auto:5"` | 自定义阈值为 5% |

三个使用要点：

- 工具少于约 10 个时全量预载反而更快——搜索本身要多一次往返，只有工具足够多、靠后续每轮更省的 context 摊平，这次往返才划算。
- 可发现性靠 name 和 description 的关键词质量（`search_slack_messages` 优于 `query_slack`）；再在 system prompt 里列一句可搜的类别（如 "You can search for tools to interact with Slack, GitHub, and Jira"），帮 agent 知道有什么可搜。
- 会话长到触发 compaction 时，已加载的工具定义可能被压缩掉——agent 会按需重新搜索，不用干预。

In [ ]:
import os
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    SystemMessage,
    AssistantMessage,
    ResultMessage,
)


async def demo_external_mcp():
    options = ClaudeAgentOptions(
        mcp_servers={
            "playwright": {"command": "npx", "args": ["@playwright/mcp@latest"]}
        },
        allowed_tools=["mcp__playwright__*"],  # 加进名单免审批；不加则每次调用走权限流程
    )
    async for message in query(
        prompt="Open example.com and describe what you see", options=options
    ):
        # init 消息里核对 MCP server 连接状态
        if isinstance(message, SystemMessage) and message.subtype == "init":
            print("MCP servers:", message.data.get("mcp_servers"))
        # 观察 Claude 实际调了哪些 MCP tool
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "name") and block.name.startswith("mcp__"):
                    print("MCP tool called:", block.name)
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_external_mcp()